# Milestone 3

**Name  : Khoizuron Syakira**\
**Batch : CODA RMT-023**

This notebook explores the data, writes down what I found for cleaning, and checks the data with Great Expectations. I do this before building the automatic pipeline (Airflow + PySpark + MongoDB).

**Scenario:** I am a Data Engineer at a travel agency. I am building a pipeline that loads travel booking data into MongoDB. Then the analysis team can study popular destinations, cancellations, and the cost of one trip.

## 1. Import Library

In [1]:
import pandas as pd  # read and work with tables
import great_expectations as gx  # data validation
import great_expectations.expectations as gxe  # groups of expectations (rules)

## 2. Load Data

In [2]:
df = pd.read_csv("/workspace/data/P2M3_khoizuron_syakira_data_raw.csv")  # read the raw file
df.head()  # show the first 5 rows

,Booking_ID,Booking_Date,Travel_Date,Return_Date,Customer_ID,Customer_Name,Customer_Age,Gender,Customer_City,Customer_Country,...,Meal_Plan,Payment_Method,Booking_Status,Cancellation_Status,Cancellation_Reason,Discount_Amount,Customer_Rating,Customer_Review_Comment,Total_Trip_Cost,Transportation_Type
0,1,2026-02-23,2026-04-18,2026-04-21,cust01,Kenzo Vaughn,18,Male,Dubai,United Arab Emirates,...,Breakfast Included,Net Banking,Completed,Not Cancelled,Not Applicable,12800,1.0,Poor service experience,115000,Flight
1,2,2025-05-30,2025-06-29,2025-07-02,cust02,Priya Carter,20,Female,Udaipur,India,...,Full Board,Net Banking,Completed,Not Cancelled,Not Applicable,0,1.5,Poor service experience,56800,Train
2,3,2024-12-30,2025-03-14,2025-03-23,cust03,Elina Vaughn,31,Female,Dubai,United Arab Emirates,...,Full Board,UPI,Completed,Not Cancelled,Not Applicable,74400,2.0,Several arrangements need improvement,669800,Flight
3,4,2026-06-17,2026-09-19,2026-09-26,cust04,Mei Ferraro,30,Male,Kathmandu,Nepal,...,Room Only,Credit Card,Confirmed,Not Cancelled,Not Applicable,4500,0.0,Not yet travelled,84700,Flight
4,5,2025-03-21,2025-04-21,2025-04-24,cust05,Alina Desai,39,Female,Kuala Lumpur,Malaysia,...,Room Only,Digital Wallet,Completed,Not Cancelled,Not Applicable,2800,3.6,Good overall experience,15600,Flight


## 3. Data Exploration

**Purpose:** understand the data first (size, data types, missing values, duplicates, etc.). This helps me decide **which cleaning steps to write in `transform.py` and which rules make sense to test with Great Expectations.**

### 3.1 Size and Data Types
**Purpose:** see how many rows and columns there are, and the data type of each column.

In [3]:
print(df.shape)  # total rows and columns
df.info()  # column names, non-empty counts, and data types

(1000, 29)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Booking_ID               1000 non-null   int64  
 1   Booking_Date             1000 non-null   object 
 2   Travel_Date              1000 non-null   object 
 3   Return_Date              1000 non-null   object 
 4   Customer_ID              1000 non-null   object 
 5   Customer_Name            1000 non-null   object 
 6   Customer_Age             1000 non-null   int64  
 7   Gender                   1000 non-null   object 
 8   Customer_City            1000 non-null   object 
 9   Customer_Country         1000 non-null   object 
 10  Destination_City         1000 non-null   object 
 11  Destination_Country      1000 non-null   object 
 12  Number_of_Travellers     1000 non-null   int64  
 13  Number_of_Adults         1000 non-null   int64  
 14  Number_of_Chil

**Insight:** the data has 1,000 rows and 29 columns. There are 9 integer columns, 2 decimal columns (`Hotel_Rating`, `Customer_Rating`), and the rest are text columns. The date columns (`Booking_Date`, `Travel_Date`, `Return_Date`) are still text.

### 3.2 Summary of Number Columns
**Purpose:** see the smallest and biggest values, so I can set reasonable limits.

In [4]:
df.describe().T  # summary of numbers: mean, min, max (.T makes it easier to read)

,count,mean,std,min,25%,50%,75%,max
Booking_ID,1000.0,500.5000,288.819436,1.0,250.75,500.5,750.25,1000.0
Customer_Age,1000.0,34.4680,9.327914,18.0,26.00,34.0,42.00,50.0
Number_of_Travellers,1000.0,5.6610,3.318900,1.0,3.00,5.0,8.00,15.0
Number_of_Adults,1000.0,4.2950,2.342106,1.0,2.00,4.0,5.00,15.0
Number_of_Children,1000.0,1.3660,1.593293,0.0,0.00,1.0,3.00,7.0
Hotel_Rating,1000.0,2.9486,1.299325,1.0,1.80,2.9,4.20,5.0
Number_of_Rooms,1000.0,2.9590,1.534821,1.0,2.00,3.0,4.00,8.0
Number_of_Nights,1000.0,5.3750,2.262188,2.0,3.00,5.0,7.00,10.0
Discount_Amount,1000.0,15980.0000,24817.542796,0.0,0.00,7850.0,20125.00,289300.0
Customer_Rating,1000.0,2.3150,1.673941,0.0,1.00,2.3,3.90,5.0


**Insight:** `Customer_Age` is about 18-50, `Hotel_Rating` is 1-5, `Number_of_Nights` is 2-10, and the average `Total_Trip_Cost` is about 227,839. `Customer_Rating` is 0-5. The 0 values need a check in part 3.8.

### 3.3 Missing Values and Duplicates
**Purpose:** check if there are empty cells or repeated rows that need cleaning.

In [5]:
print(df.isna().sum().sum())  # total empty cells in the table
print(df.duplicated().sum())  # total repeated rows

0
0


**Insight:** there are no missing values (0) and no duplicate rows (0).

### 3.4 Category Columns
**Purpose:** see which values are inside the category columns, check that the same value is not written in two ways (for example “Male” and “male”), and check that the text columns have no extra spaces.

In [6]:
for col in ["Gender", "Meal_Plan", "Booking_Status", "Cancellation_Status", "Transportation_Type"]:  # category columns to check
    print(df[col].value_counts(), "\n")  # show each value and how many times it appears
    
for col in df.select_dtypes("object").columns: # all text columns
       print(col, (df[col] != df[col].str.strip()).sum())  # rows with extra spaces

Gender
Female    568
Male      432
Name: count, dtype: int64 

Meal_Plan
Breakfast Included    392
Room Only             220
Full Board            208
Half Board            180
Name: count, dtype: int64 

Booking_Status
Completed    772
Cancelled    141
Confirmed     79
Pending        8
Name: count, dtype: int64 

Cancellation_Status
Not Cancelled    859
Cancelled        141
Name: count, dtype: int64 

Transportation_Type
Flight    682
Train     186
Car       132
Name: count, dtype: int64 

Booking_Date 0
Travel_Date 0
Return_Date 0
Customer_ID 0
Customer_Name 0
Gender 0
Customer_City 0
Customer_Country 0
Destination_City 0
Destination_Country 0
Hotel_Name 0
Meal_Plan 0
Payment_Method 0
Booking_Status 0
Cancellation_Status 0
Cancellation_Reason 0
Customer_Review_Comment 0
Transportation_Type 0


**Insight:** the values are written in a consistent way. `Booking_Status` has only 4 values (Completed, Cancelled, Confirmed, Pending), `Meal_Plan` has 4 values, and `Gender` has 2 values (Male and Female).All 18 text columns have 0 rows with extra spaces at the start or end, so the text is already clean. I still keep the trim step in transform.py as a  safety step, because the data in the next scheduled runs can be different and may need to be cleaned.

### 3.5 Date Format and Date Logic
**Purpose:** make sure the dates can be read and make sense (the return date cannot be before the travel date).

In [7]:
booking = pd.to_datetime(df["Booking_Date"], format="%Y-%m-%d", errors="coerce")  # change text to date with a fixed format (if it fails, the result is empty)
travel = pd.to_datetime(df["Travel_Date"], format="%Y-%m-%d", errors="coerce")  # travel (departure) date
back = pd.to_datetime(df["Return_Date"], format="%Y-%m-%d", errors="coerce")  # return date
print(booking.isna().sum(), travel.isna().sum(), back.isna().sum())  # how many dates could not be read
print((back < travel).sum())  # bookings that return before the travel date
print((travel < booking).sum())  # bookings that travel before the booking date

0 0 0
0
0


**Insight:** all dates can be read with the format %Y-%m-%d (for example 2026-02-23) and the order makes sense (all results are 0). I only need to change the type from text to date.

### 3.6 Checking Columns Against Each Other
**Purpose:** check if the numbers in different columns match.

In [8]:
print((df["Number_of_Travellers"] != df["Number_of_Adults"] + df["Number_of_Children"]).sum())  # total travellers must = adults + children
print(((back - travel).dt.days != df["Number_of_Nights"]).sum())  # number of nights must = date difference
print((df["Discount_Amount"] >= df["Total_Trip_Cost"]).sum())  # bookings where the discount is not smaller than the trip cost

0
0
0


**Insight:** all three results are 0, so the data is consistent. The number of nights already equals the date difference, so I do not need to create a new duration column. The discount is always smaller than the trip cost (result is 0), so the discount never goes above the price of the trip.

### 3.7 ID Columns and Column Name Rules
**Purpose:** make sure the IDs are not repeated (this is the base for the *unique* expectation) and the column names follow the assignment rules.

In [9]:
print(df["Booking_ID"].is_unique)  # True if no Booking_ID is repeated
print(df["Customer_ID"].is_unique)  # True if no Customer_ID is repeated
print(df["Customer_ID"].str.match(r"^cust\d+$").all())  # True if every ID follows the pattern 'cust' + number
print([c for c in df.columns if c == c.lower()])  # column names that are all lowercase (should be empty)

True
True
True
[]


**Insight:** `Booking_ID` and `Customer_ID` are unique, every `Customer_ID` follows the pattern `cust` + number, and no column name is all lowercase, so the raw data follows the dataset rule. For the pipeline, I change all column names to lowercase snake_case in transform.py, so they are consistent in MongoDB.

### 3.8 Value 0 in Customer_Rating
**Purpose:** check if rating 0 is a real score or just means "no rating yet".

In [10]:
print(df.groupby("Booking_Status")["Customer_Rating"].agg(["count", "min", "mean", "max"]))  # rating summary for each booking status
print((df["Customer_Rating"] == 0).sum())  # how many ratings are 0

                count  min      mean  max
Booking_Status                           
Cancelled         141  0.0  0.000000  0.0
Completed         772  1.0  2.998705  5.0
Confirmed          79  0.0  0.000000  0.0
Pending             8  0.0  0.000000  0.0
228


**Insight:** rating 0 appears in 228 rows. These are all the bookings with status Cancelled (141), Confirmed (79), and Pending (8). Bookings with status Completed always have a rating from 1 to 5 (average 3.0). So rating 0 really means **"no rating yet"**, because the trip was not taken or was cancelled. It is not a score from the customer. If I leave it, the overall average rating drops to about 2.3, but the real average is 3.0.

### 3.9 Trip Cost by Booking Status

**Purpose:** checking if the bookings that are not completed (Cancelled, Confirmed, Pending) still have a trip cost, because this changes how the analysis team must calculate the cost of one trip.

In [11]:
print(df.groupby("Booking_Status")["Total_Trip_Cost"].mean())  # average trip cost for each booking status
print(df.groupby("Booking_Status")["Total_Trip_Cost"].median())  # middle trip cost for each booking status
print(df.groupby("Booking_Status")["Total_Trip_Cost"].count())  # how many bookings in each status

Booking_Status
Cancelled    245278.723404
Completed    220195.595855
Confirmed    272178.481013
Pending      220187.500000
Name: Total_Trip_Cost, dtype: float64
Booking_Status
Cancelled    171700.0
Completed    150950.0
Confirmed    179000.0
Pending      197800.0
Name: Total_Trip_Cost, dtype: float64
Booking_Status
Cancelled    141
Completed    772
Confirmed     79
Pending        8
Name: Total_Trip_Cost, dtype: int64


**Insight:** all booking statuses have a trip cost, so the cost does not become 0 when a booking is cancelled. The `Cancelled bookings (141)` still have an average cost of about 245,279 and a middle cost of 171,700. Both are higher than the `Completed bookings (average about 220,196 and middle cost 150,950)`. In every status the average is higher than the middle cost, so a few expensive trips pull the average up, the same as I found in part 3.2.

This is important because when the analysis team calculates the cost of one trip, they need to filter by `Booking_Status` first, so cancelled and not yet travelled bookings do not make the result wrong. The Completed bookings (772) are the biggest group therefore they are the safest group base for the cost of one trip.

The Pending group has only 8 bookings, so I do not use it to compare. The data itself does not show why the Cancelled bookings cost more, so I do not make a conclusion about it.

## 4. Conclusion of the Data Exploration

The data has 1,000 rows and 29 columns. There are no missing values or duplicates, the dates and numbers match across columns, and the IDs are unique. The only problem I found is rating 0, which really means "no rating yet".

Cleaning and processing steps that I will do in `transform.py`:
1. Drop `Customer_Name` because it is private data and not needed for analysis.
2. Remove extra spaces in text columns as a safety steps.
3. Remove duplicates and empty values as a safety step, because new data in the next scheduled runs can be different.
4. Change `Customer_Rating` value 0 to empty (null), so the average rating is not misleading.
5. Change `Booking_Date`, `Travel_Date`, and `Return_Date` from text to date type.
6. Add the column `Cost_Per_Traveller` (`Total_Trip_Cost` divided by `Number_of_Travellers`) to compare the cost between trips.
7. Change all column names to lowercase with underscore (snake_case).

## 5. Data Validation with Great Expectations

The rules I test come from the exploration above:

| No | Expectation | Column | Reason from the exploration |
|---|---|---|---|
 1 | to be unique | `Booking_ID` | 3.7: IDs are unique, so each booking is counted once and popular destinations and cancellation rates are not inflated by duplicate rows |
| 2 | to be between | `Hotel_Rating` (1-5) | 3.2: hotel rating scale is 1 to 5, so a rating outside the scale does not give a wrong average rating |
| 3 | to be in set | `Booking_Status` | 3.4: only 4 known statuses, so the cancellation rate does not miss a new or misspelled status (for example "Canceled") |
| 4 | to be in type list | `Total_Trip_Cost` | 3.1: cost must be a number, so the average and total cost of a trip can be calculated |
| 5 | mean to be between | `Customer_Rating` (Completed) | 3.8: the real average rating is 3.0, because rating 0 only means "no rating yet" in the other statuses |
| 6 | median to be between | `Total_Trip_Cost` (Completed) | 3.9: the cost is also high for the Cancelled bookings, and a few expensive trips pull the average up |
| 7 | column pair A to be greater than B | `Total_Trip_Cost`, `Discount_Amount` | 3.6: the discount is always smaller than the trip cost, so the cost analysis is not wrong |

In [12]:
context = gx.get_context()  # create the Great Expectations context
data_source = context.data_sources.add_pandas("travel_source")  # pandas data source
data_asset = data_source.add_dataframe_asset(name="travel_df")  # data asset (a DataFrame)
batch_definition = data_asset.add_batch_definition_whole_dataframe("whole_df")  # use the whole DataFrame
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})  # the batch of data to validate

### Expectation 1 - `to be unique`

`Booking_ID` must be unique because one row is one booking. If a booking appears twice, popular destinations and the cancellation rate would be counted wrong.

In [13]:
batch.validate(gxe.ExpectColumnValuesToBeUnique(column="Booking_ID"))  # booking ID must not repeat

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 1778.75it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_unique",
    "kwargs": {
      "batch_id": "travel_source-travel_df",
      "column": "Booking_ID"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 1000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### Expectation 2 - `to be between`

`Hotel_Rating` must be on a scale from 1 to 5. A value outside this scale would give a wrong average hotel rating.

In [14]:
batch.validate(gxe.ExpectColumnValuesToBeBetween(column="Hotel_Rating", min_value=1, max_value=5))  # rating scale

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 1772.14it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "travel_source-travel_df",
      "column": "Hotel_Rating",
      "min_value": 1.0,
      "max_value": 5.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 1000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### Expectation 3 - `to be in set`

`Booking_Status` can only have the four known statuses. The cancellation rate depends on these exact values, so a new or misspelled status must be caught.

In [15]:
batch.validate(gxe.ExpectColumnValuesToBeInSet(column="Booking_Status", value_set=["Completed", "Confirmed", "Cancelled", "Pending"]))  # status values

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 1523.93it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_in_set",
    "kwargs": {
      "batch_id": "travel_source-travel_df",
      "column": "Booking_Status",
      "value_set": [
        "Completed",
        "Confirmed",
        "Cancelled",
        "Pending"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 1000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### Expectation 4 - `to be in type list`

`Total_Trip_Cost` must be a number. If it is text, the average and total cost of a trip cannot be calculated.

In [16]:
batch.validate(gxe.ExpectColumnValuesToBeInTypeList(column="Total_Trip_Cost", type_list=["int64", "float64"]))  # data type

Calculating Metrics: 100%|██████████| 1/1 [00:00<00:00, 422.00it/s] 


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_values_to_be_in_type_list",
    "kwargs": {
      "batch_id": "travel_source-travel_df",
      "column": "Total_Trip_Cost",
      "type_list": [
        "int64",
        "float64"
      ]
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": "int64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### Batch for the Completed Bookings

The cost and the rating of a booking depend on its status (explained on parts 3.8 and 3.9). So for Expectation 5 and 6, I make a second batch that has only the Completed bookings (772 rows).

In [17]:
df_completed = df[df["Booking_Status"] == "Completed"]  # keep only the completed bookings
batch_completed = batch_definition.get_batch(batch_parameters={"dataframe": df_completed})  # make a new batch from these rows

### Expectation 5 - `mean to be between`

The average of `Customer_Rating` for the Completed bookings must be between 2.5 and 3.5. In part 3.8 I found that the real average is 3.0, and rating 0 (no rating yet) only appears in the other statuses. If many `Completed` bookings get a wrong rating (for example 0) in the next scheduled runs, the average would go down and this check would catch it.

In [18]:
batch_completed.validate(gxe.ExpectColumnMeanToBeBetween(column="Customer_Rating", min_value=2.5, max_value=3.5))  # average rating of the completed bookings

Calculating Metrics: 100%|██████████| 4/4 [00:00<00:00, 1327.21it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_mean_to_be_between",
    "kwargs": {
      "batch_id": "travel_source-travel_df",
      "column": "Customer_Rating",
      "min_value": 2.5,
      "max_value": 3.5
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": 2.9987046632124352
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### Expectation 6 - `median to be between`

The median of `Total_Trip_Cost` for the Completed bookings must be between 120,000 and 200,000. I only use the Completed bookings because the Cancelled bookings also have a high trip cost, so mixing all statuses would give a wrong calculation of cost for one trip. I use the median and not the average because a few very expensive trips pull the average up.

In [19]:
batch_completed.validate(gxe.ExpectColumnMedianToBeBetween(column="Total_Trip_Cost", min_value=120000, max_value=200000))  # middle cost of a completed trip

Calculating Metrics: 100%|██████████| 4/4 [00:00<00:00, 1363.11it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_median_to_be_between",
    "kwargs": {
      "batch_id": "travel_source-travel_df",
      "column": "Total_Trip_Cost",
      "min_value": 120000.0,
      "max_value": 200000.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "observed_value": 150950.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### Expectation 7 - `column pair A to be greater than B`

`Total_Trip_Cost` must be bigger than `Discount_Amount` in every booking. A discount cannot be bigger than the price of the trip, whatever the status, so I check all bookings.

In [20]:
batch.validate(gxe.ExpectColumnPairValuesAToBeGreaterThanB(column_A="Total_Trip_Cost", column_B="Discount_Amount"))  # trip cost must be bigger than the discount in every row

Calculating Metrics: 100%|██████████| 9/9 [00:00<00:00, 1312.09it/s]


{
  "success": true,
  "expectation_config": {
    "type": "expect_column_pair_values_a_to_be_greater_than_b",
    "kwargs": {
      "batch_id": "travel_source-travel_df",
      "column_A": "Total_Trip_Cost",
      "column_B": "Discount_Amount"
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 1000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0,
    "partial_unexpected_counts": [],
    "partial_unexpected_index_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

## 6. Conclusion

**Data exploration:** the data has 1,000 rows and 29 columns. There are no missing values, no duplicate rows, and no extra spaces in the 18 text columns. The dates can be read and are in the right order, the numbers match across columns, and the IDs are unique. The only problem I found is `Customer_Rating` value 0 (228 rows), which really means "no rating yet" for the Cancelled (141), Confirmed (79), and Pending (8) bookings. The real average rating is 3.0, not 2.3.

**Cleaning in `transform.py`:** based on this, I drop `Customer_Name`, remove extra spaces, remove duplicates and empty values (both as safety steps), change rating 0 to null, change the date columns to date type, add `Cost_Per_Traveller`, and change all column names to lowercase snake_case.

**Data validation:** all 7 expectations in Great Expectations passed (`success: true`), so the data is ready to be cleaned in `transform.py` and loaded to MongoDB.

**Suggestions for the next development:**
1. Validate the data again after `transform.py`. Now I only test the raw data, so I do not know yet if the cleaning worked (for example rating 0 is now null and `Customer_Name` is gone).
2. Check the outliers in `Total_Trip_Cost`. The biggest trip (1,639,200) is far from the middle value (157,200), so I need to check if these trips are real or input mistakes before the analysis team uses the average.
3. Filter by `Booking_Status` before the analysis team calculates the cost of one trip, because the 141 Cancelled bookings still have a trip cost (part 3.9).
4. Make the range in Expectation 6 wider, because 120,000 to 200,000 is based on this dataset only.